In [42]:
import numpy as np
import pandas as pd
import re
import torch
import torch.nn as nn
import torch.optim as optim
import math
import datetime

from torch.optim import SGD, Optimizer
from torch.utils.data import TensorDataset
from tqdm.notebook import tqdm
from tensorboard import notebook
from torch.utils.tensorboard import SummaryWriter

In [21]:
%reload_ext tensorboard
%tensorboard --logdir tb_logs

In [31]:
torch.cuda.is_available()
device = torch.device("cpu")
# Give each run its own subdirectory so TensorBoard shows separate, distinctly colored runs
run_name = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
writer = SummaryWriter(f"tb_logs/{run_name}")

In [32]:
#data = pd.read_csv("/home/hutchii/projects/MentalHealthLLM/data/mh_sentiment_data.csv")
data = pd.read_csv("/Users/hutchii/projects/MentalHealthLLM/data/mh_sentiment_data.csv")

In [54]:
NUM_EPOCH = 5
MAX_SEQ_LEN = 200
PAD_TOKEN = "<PAD>"
BLOCK_SIZE = 64
NUM_DATA = len(statements)
SPLIT_OFFSET = round(NUM_DATA * .75)

statements = data["statement"]
targets = data["status"]

39782


In [55]:
words = (
    data["statement"]
    .dropna()
    .str.lower()
    .str.findall(r"[a-z]+(?:'[a-z]+)*")
    .explode()
    .dropna()
)

conditions = (
    data["status"]
    .unique()
)

target_dict = {s:i for i,s in enumerate(conditions)}
print(target_dict)


unique = list(set(words))
sorted_words = sorted(unique)

stoi = {s:i for i,s in enumerate(sorted_words, start=1)}
itos = {i:s for i,s in enumerate(sorted_words, start=1)}

stoi[PAD_TOKEN] = 0
itos[0] = PAD_TOKEN

embeddings_dict = nn.Embedding(len(stoi.values()), 768).to(device)

{'Anxiety': 0, 'Normal': 1, 'Depression': 2, 'Suicidal': 3, 'Stress': 4, 'Bipolar': 5, 'Personality disorder': 6}


In [56]:
# Create 
x_train = statements[:SPLIT_OFFSET]
y_train = targets[:SPLIT_OFFSET]
print("TEST")
print(x_train.shape)
print(y_train.shape)
print("\n")

print("VALIDATION")
x_val = statements[SPLIT_OFFSET:]
y_val = targets[SPLIT_OFFSET:]
print(x_val.shape)
print(y_val.shape)

TEST
(39782,)
(39782,)


VALIDATION
(13261,)
(13261,)


In [39]:
def encode(sentence:str) -> list[int]:
    cleaned = re.findall(r"[a-z]+(?:'[a-z]+)*", sentence.lower())
    string_len = len(cleaned)

    if string_len > MAX_SEQ_LEN:
        s = cleaned[:MAX_SEQ_LEN]
    else:
        num_padding = MAX_SEQ_LEN - string_len
        padding = [PAD_TOKEN for _ in range(num_padding)]
        s = cleaned + padding

    return [stoi[w] for w in s]

def decode(sequence:list[int]) -> str:
    return "".join([itos[i] for i in sequence if i != 0])

def encode_targets(targets):
  return [target_dict[s] for s in targets]

def get_sample_batch(statements):
  index = 0
  while index < len(statements):
    yield statements[index:index+BLOCK_SIZE]
    index += BLOCK_SIZE

def get_target_batch(targets):
  index = 0
  while index < len(targets):
    yield targets[index:index+BLOCK_SIZE]
    index += BLOCK_SIZE

In [40]:
class MentalModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(768, 512),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, 7)
    )

  def forward(self,x):
    return self.model(x)

In [ ]:
#Create Tensor dataset for train and test
x_train =
y_train =
train_dataset = TensorDataset(x_train, y_train )

In [41]:
model = MentalModel().to(device)
# Switching to Adam for better stability in text tasks
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)
loss_fn = nn.CrossEntropyLoss()



global_step = 0
for e in tqdm(range(NUM_EPOCH), desc="Epochs"):
  # Shuffle each epoch (statements/targets together) so batches aren't dominated by one class
  perm = np.random.permutation(len(statements))
  shuffled_statements = statements.iloc[perm].reset_index(drop=True)
  shuffled_targets = data["status"].iloc[perm].reset_index(drop=True)

  sample_generator = get_sample_batch(shuffled_statements)
  target_generator = get_target_batch(shuffled_targets)

  for batch in tqdm(sample_generator, desc=f"Epoch {e+1} Batches", total=math.ceil(len(statements)/BLOCK_SIZE)):
    # Get current batch
    x_batch = torch.tensor(np.array([encode(str(s)) for s in batch])).to(device)
    x_embed = embeddings_dict(x_batch)
    x_pooled = torch.mean(input=x_embed, dim=1)

    # Get targets for current batch
    y_batch = next(target_generator)
    y_encoded = torch.tensor(np.array(encode_targets(y_batch))).to(device)

    model.train()
    optimizer.zero_grad()

    predictions = model(x_pooled)
    loss = loss_fn(predictions, y_encoded)

    # Log to TensorBoard
    writer.add_scalar("Loss/train", loss.item(), global_step)

    loss.backward()
    optimizer.step()
    global_step += 1

  writer.flush()

Epochs:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 1 Batches:   0%|          | 0/829 [00:00<?, ?it/s]

Epoch 2 Batches:   0%|          | 0/829 [00:00<?, ?it/s]

Epoch 3 Batches:   0%|          | 0/829 [00:00<?, ?it/s]

Epoch 4 Batches:   0%|          | 0/829 [00:00<?, ?it/s]

Epoch 5 Batches:   0%|          | 0/829 [00:00<?, ?it/s]